# Fine-Tuning Llama 3 Using Azure Machine Learning Notebooks and Hugging Face Tools

## Introduction

In this lab we will fine-tune a Llama 3 model using tools, models and datasets from Hugging Face on an Azure Machine Learning (AML) GPU instance.  In particular, we will perform Low-Rank Approximation (LoRA) training which can run on even a smaller GPU such as the T4 using the [PyTorch](https://pytorch.org/) deep-learning framework.

We will fine-tune the model using a dataset consisting of medical questions and answers.  In essence, we will be creating a 'Llama 3 Doctor' model that has specific knowledge of the medical domain.  

## Prerequisites
To run this lab you need to have the following:
* An AML GPU-based compute instance
* A conda environment named `llama3_ft`, created using the enclosed `1_setup_conda.sh` file
* The [Medical Dialogs](https://huggingface.co/datasets/ruslanmv/ai-medical-chatbot) dataset, loaded into AML's Data Repository, accessible through the Assets->Data link on the left-hand side of the screen, with the name `MedicalDialogs`.

## Tools Used
The Python tools used in this lab are the following open-source Hugging Face tools:

* [Transformers](https://huggingface.co/docs/transformers/v4.17.0/en/index) - Implementation of a number of deep-learning models using the Transformer architecture
* [PEFT](https://huggingface.co/docs/peft/index) - Implementation of Parameter-Efficient Fine-Tuning, which allows the fine-tuning of pretrained models using only a small subset of their parameters.  We will be using the Quantized LORA (QLORA) algorithm for fine-tuning a model that was quantized to use 4 bits for each weight instead of 16 bits. 
* [TRL](https://huggingface.co/docs/trl/index) - This library contains a number of algorithms that help train Transformer-based language models using Reinforcement Learning.  As our dataset contains medical questions and answers, we will be using the Supervised Fine-Tuning (SFT) algorithm.
* [Accelerate](https://huggingface.co/docs/accelerate/index) - This library makes it easy to run multi-gpu training and is integrated into the other libraries we will use.
* [BitsAndBytes](https://huggingface.co/docs/bitsandbytes/index) - This library provides tools for loading models in a quantized form for PyTorch. 



In [33]:
# load the $HOME environment variable
import os
home = os.environ["HOME"]

# set huggingface model cache
# define the default cache path for transformers
# subfolder huggingface/hub/modelfolder will be created automatically
os.environ["XDG_CACHE_HOME"] = f"{home}/Code/MODELS/"

In [34]:
# Working directory on the AML compute instance
!pwd

/Users/yingding/Code/VCS/ai/model-fine-tuning/10-local-mps


## Imports & Definitions
In this section we will import the classes we need and setup some definitions to be used later on.

In [35]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import (
    LoraConfig,
    PeftModel,
    AutoPeftModelForCausalLM,
    prepare_model_for_kbit_training,
    get_peft_model,
)

import os
import torch

from datasets import load_dataset, Dataset, ReadInstruction
from trl import SFTTrainer, setup_chat_format, SFTConfig

cwd = os.getcwd()
print(f"Current working directory: {cwd}")


Current working directory: /Users/yingding/Code/VCS/ai/model-fine-tuning/10-local-mps


In [36]:
# Name of base model to use for fine-tuning, retrieved from HuggingFace
base_model = "NousResearch/Meta-Llama-3-8B"

# Name and version of dataset, to be retrieved from AML's Data Repository
# finetune_dataset = "MedicalDialogs"
finetune_dataset = "ruslanmv/ai-medical-chatbot"
finetune_dataset_version = "1"

# How many samples to use from the dataset
finetune_dataset_samples = 50 # 1000

# Size of test (evaluation) set, of the total number of samples
test_size = 0.1

model_dir = os.path.join(cwd, "models")
if not os.path.exists(model_dir):
    os.makedirs(model_dir)
print(f"Model directory: {model_dir}")

# Directory name in which to save the LoRA adapters for the fine-tuned model
finetuned_model = "llama3-8b-chat-doctor"

# Directory name in which to save the configuration settings for the fine-tuned model
finetuned_model_config = f"{finetuned_model}_config"

# Directory name in which to save the full fine-tuned model
finetuned_model_full = f"{finetuned_model}_full"

# Directory in which to save model files downloaded from HuggingFace
# cache_dir = "/mnt/tmp/hf_cache"

# Sample question to ask the final fine-tuned model
messages = [
    {
        "role": "user",
        "content": "Hello doctor, I get red blotches on my skin whenever I'm next to a cat.  What can I do?"
    }
]

Model directory: /Users/yingding/Code/VCS/ai/model-fine-tuning/10-local-mps/models


## Load the Model
In this section we load the model and tokenizer, performing the 4-bit quantization and moving the model to the GPU.  In addition, we setup the model and tokenizer to automatically add the chat template to the data.  The _chat template_ is the addition of custom markup tokens so the model is able to distinguish between roles, content and other types of information.

In [37]:

# Setup 4-bit quantization 
# testing cuda
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    qlora_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    args = {
        "qunatization_config": qlora_config,
        "device_map": "auto",
    }
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using Apple Silicon GPU")
    args = {
        "device_map": device,
    }
else:
    args = {
        "device_map": "auto",
    }

# Download the tokenizer from HuggingFace
tokenizer = AutoTokenizer.from_pretrained(base_model)

# Download the model from HuggingFace
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    # Automatically load the model into the GPU
    # device_map="auto",
    attn_implementation="eager",
    # Apply quantization settings if specified
    **args,
    # cache_dir=cache_dir
)

# Ensure the model and tokenizer are setup to use the proper prefixes and suffixes required for 
# marking the role and content 
model, tokenizer = setup_chat_format(model, tokenizer)

Using Apple Silicon GPU


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

## Setup LoRA 
In this section we setup the LoRA configuration, and prepare the model for parameter-efficient fine-tuning.

In [38]:
peft_config = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM", 
)
model = get_peft_model(model, peft_config)

## Retrieve and Prepare the Fine-Tuning Data
In this section we:
* Retrieve the medial dialogs dataset from AML's Data Repository
* Sample from the dataset
* Split the data into training and test (evaluation) sets


In [39]:
cwd = os.getcwd()
data_file_dir = os.path.join(cwd, "data")
hf_cache = data_file_dir
num_data_rows = 1000

In [40]:
# the cache_dir folder will be created automatically
hf_dataset = load_dataset(finetune_dataset, split="all", cache_dir=hf_cache)
# hf_dataset = hf_dataset.shuffle(seed=65).select(range(num_data_rows))

print(hf_dataset)

# load huggingface dataset to a pandas dataframe
df = hf_dataset.to_pandas()

print(df.size)
# display a sample of the dataset
df.head()

Dataset({
    features: ['Description', 'Patient', 'Doctor'],
    num_rows: 256916
})
770748


,Description,Patient,Doctor
0,Q. What does abutment of the nerve root mean?,"Hi doctor,I am just wondering what is abutting...",Hi. I have gone through your query with dilige...
1,Q. What should I do to reduce my weight gained...,"Hi doctor, I am a 22-year-old female who was d...",Hi. You have really done well with the hypothy...
2,Q. I have started to get lots of acne on my fa...,Hi doctor! I used to have clear skin but since...,Hi there Acne has multifactorial etiology. Onl...
3,Q. Why do I have uncomfortable feeling between...,"Hello doctor,I am having an uncomfortable feel...",Hello. The popping and discomfort what you fel...
4,Q. My symptoms after intercourse threatns me e...,"Hello doctor,Before two years had sex with a c...",Hello. The HIV test uses a finger prick blood ...


In [26]:
# from azure.ai.ml import MLClient
# from azure.identity import DefaultAzureCredential
# import pandas as pd

# credential = DefaultAzureCredential()
# SUBSCRIPTION="eba489bb-ec02-466c-9806-a269d915d943"
# RESOURCE_GROUP="yuvmaz-aml"
# WS_NAME="yuvmaz-aml"

# ml_client = MLClient(
#     credential=credential,
#     subscription_id=SUBSCRIPTION,
#     resource_group_name=RESOURCE_GROUP,
#     workspace_name=WS_NAME
# )

# # Retrieve the dataset from AML where it is stored in Parquet format
# data_asset = ml_client.data.get(name=finetune_dataset, version=finetune_dataset_version)
# df = pd.read_parquet(data_asset.path)

# Display a sample of the dataset 
# df.head()

In [ ]:
# Sample the required number of data points - retrieve the 'Doctor' and 'Patient' fields from the dataset
dataset = Dataset.from_pandas(df[['Patient', 'Doctor']].sample(finetune_dataset_samples))

# Format the dataset - the Patient field is passed as the user query, while the Doctor field is the answer we expect
# From the model
def format_chat_template(row):
    row_json = [{"role": "user", "content": row["Patient"]},
               {"role": "assistant", "content": row["Doctor"]}]
    row["text"] = tokenizer.apply_chat_template(row_json, tokenize=False)
    return row


dataset = dataset.map(
    format_chat_template,
    num_proc=4,
)

# Split the dataset into training and test sets
dataset = dataset.train_test_split(test_size=test_size)

# Show the resulting dataset and split sizes
dataset

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['Patient', 'Doctor', '__index_level_0__', 'text'],
        num_rows: 900
    })
    test: Dataset({
        features: ['Patient', 'Doctor', '__index_level_0__', 'text'],
        num_rows: 100
    })
})

In [28]:
# Show a sample data instance and the resulting markup to be fed to the model while fine-tuning
from pprint import pprint
pprint(dataset['train'][0])

{'Doctor': 'Hi. I can understand your concern. From history, I hope you are '
           'telling that you (male) is B positive and your fiance (female) is '
           'B negative. It will not cause a problem in first pregnancy but in '
           'second pregnancy severe hemolytic disease of the newborn can '
           'occur. So, we have to prevent this sensitization when she will be '
           'pregnant. So, just keep this in mind and in first pregnancy of '
           'your wife (she is having a negative group as per history), she '
           'should be given Rh(D) immunoglobulin injections in the antenatal '
           'period at 28 week of gestation. In addition, before marriage, you '
           'both should be investigated for thalassemia minor status. For '
           'further information consult a hematologist online -->',
 'Patient': 'Hi doctor,My blood group is B positive. I am going to marry a '
            'person with B negative blood group. Is there any problem?',


## Setup and Run the Training
In this section we:

* Perform the training, using HuggingFace TRL's Supervised Fine-Tuning (SFT) algorithm
* Send a sample query to the fine-tuned model and observer the output
* Save the LoRA adapter, the configuration and the full (merged) model

In [29]:
output_dir = os.path.join(cwd, "tmp")
if output_dir and not os.path.exists(output_dir):
    os.makedirs(output_dir)

In [42]:
training_arguments = SFTConfig(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    num_train_epochs=1,
    eval_strategy="steps",
    eval_steps=0.2,
    logging_steps=50,
    warmup_steps=10,
    logging_strategy="steps",
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    group_by_length=True,
    # report_to='azure_ml',   
    # max_seq_length=512,
    dataset_text_field="text",
    packing= False,
)

In [43]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"].remove_columns(["__index_level_0__"]),
    eval_dataset=dataset["test"].remove_columns(["__index_level_0__"]),
    peft_config=peft_config,
    processing_class=tokenizer,
    # tokenizer=tokenizer,
    # args=training_arguments,
)

/Users/yingding/Code/VENV/mpsft3.12pip/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

In [ ]:
stats = trainer.train()

/Users/yingding/Code/VENV/mpsft3.12pip/lib/python3.12/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss


In [ ]:
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", padding=True, max_length=512, truncation=True).to(device)

outputs = model.generate(**inputs, max_length=150, num_return_sequences=1)
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text.split("assistant")[1])

AssertionError: Torch not compiled with CUDA enabled

In [ ]:
trainer.save_model(finetuned_model)
peft_config.save_pretrained(finetuned_model_config)
model.merge_and_unload().save_pretrained(finetuned_model_full)